# LLM Zoomcamp HW5 

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
print("Groq API Key loaded:", "GROQ_API_KEY" in os.environ)

Groq API Key loaded: True


In [10]:
from starter import index, client

In [6]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("llm-zoomcamp")

In [26]:
import time

class RAGTraced(RAGBase):
    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search") as span:
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            # 1. Start the timer
            start_time = time.perf_counter()
            
            response = super().llm(prompt)
            
            # 2. Stop the timer and calculate duration in milliseconds
            end_time = time.perf_counter()
            duration_ms = (end_time - start_time) * 1000
            print(f"\n>>> [LLM Call Duration]: {duration_ms:.2f} ms")
            
            # Retrieve token usage
            input_tokens = response.usage.input_tokens
            output_tokens = response.usage.output_tokens
            cost = (input_tokens * 0.15 + output_tokens * 0.60) / 1_000_000
            
            # Set OpenTelemetry attributes
            span.set_attribute("input_tokens", input_tokens)
            span.set_attribute("output_tokens", output_tokens)
            span.set_attribute("cost", cost)
            return response

    def rag(self, query):
        with tracer.start_as_current_span("rag") as span:
            return super().rag(query)

# Initialize traced RAG instance
rag_traced = RAGTraced(index=index, llm_client=client, model="llama-3.3-70b-versatile")

In [27]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_traced.rag(query)
print("\n--- Answer ---\n", answer)

{
    "name": "search",
    "context": {
        "trace_id": "0x133874737fe6f8311a050f3c8a742cb2",
        "span_id": "0xf0e8638ce710d5a5",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xd276a90e388911e8",
    "start_time": "2026-07-20T12:58:37.332323Z",
    "end_time": "2026-07-20T12:58:37.334462Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "bb64d322-551d-4f4c-a262-0e222034c4a0",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}

>>> [LLM Call Duration]: 9700.00 ms
{
    "name": "llm",
    "context": {
        "trace_id": "0x133874737fe6f8311a050f3c8a742cb2",
        "span_id": "0xbe9b2ec3f5eadc13",
        "trace_stat

In [28]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult

class SQLiteSpanExporter(SpanExporter):
    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

sqlite_exporter = SQLiteSpanExporter("traces.db")
provider.add_span_processor(SimpleSpanProcessor(sqlite_exporter))

In [29]:
# Run the query 3 more times to populate the DB
for i in range(2, 5):
    rag_traced.rag(query)
    
sqlite_exporter.shutdown()

# Read and analyze the SQLite database
import pandas as pd
conn = sqlite3.connect("traces.db")
df = pd.read_sql_query("SELECT * FROM spans", conn)
conn.close()

# Q4 Spans
print("Unique spans in database:", df['name'].unique())

# Q5 Durations
df['duration_ms'] = (df['end_time'] - df['start_time']) / 1_000_000
print(df[df['name'] != 'rag'].groupby('name')['duration_ms'].sum())

# Q6 Input tokens
print("Input tokens per run:\n", df[df['name'] == 'llm']['input_tokens'])

{
    "name": "search",
    "context": {
        "trace_id": "0x48bb757d59a87e5370a09dd35a5ee92e",
        "span_id": "0x90d751d0d5429ff2",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x453c0ad61fb92b9c",
    "start_time": "2026-07-20T12:59:57.950745Z",
    "end_time": "2026-07-20T12:59:57.953101Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "bb64d322-551d-4f4c-a262-0e222034c4a0",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}

>>> [LLM Call Duration]: 656.90 ms
{
    "name": "llm",
    "context": {
        "trace_id": "0x48bb757d59a87e5370a09dd35a5ee92e",
        "span_id": "0x1b483b3783f769dd",
        "trace_state